# AI Herding · Next Step 1: Order-size limits

**For Jupyter Notebook / JupyterLab with Python 3.11 or later. Run the 7 code cells in order.**

This notebook addresses one question: when the configured per-order limit is 1000 shares, does the system actually reject oversized orders?

The notebook will:
1. Download the code version used in the previous report.
2. Inspect the original behavior with manually constructed orders.
3. Add an oversized-order rejection rule in a separate test copy and connect the scenario configuration to the actual validation entry point.
4. Check the patched behavior, cash/share commitments, and the three existing matching regression tests.
5. Save per-order CSV results and a code patch.

**No API key is needed, and GPT or other models are not called.** Downloading code and missing Python dependencies requires network access; the order-test subprocess blocks network connections.

The checks cover order creation, initial validation, and resource commitment. An accepted order is usually in the `committed` state; acceptance does not mean execution. This notebook does not run a new 15-round LLM market experiment.

Source: the team's [Yuhan branch](https://github.com/yuuhanjin/AI_herding_experiment/tree/Yuhan), pinned to `074faff3c8f6255c917d2a2593a5b476c947ca84`.

Saved execution outputs are retained below. Running the notebook again creates working directories on the local computer.


## Code cell 1/7: Check the environment and create the working directory

Select the code cell and press `Shift + Enter`. Continue after the environment check completes.


In [1]:
import sys
import os
import json
import csv
import hashlib
import subprocess
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError("Switch to a Python 3.11 or later kernel in Jupyter before running this notebook.")

WORK_DIR = Path.cwd() / "AI_Herding_Step1_Work"
WORK_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR = WORK_DIR / "results"
RESULT_DIR.mkdir(exist_ok=True)

print("Python：", sys.version.split()[0])
print("Working directory: ", WORK_DIR.resolve())
print("Environment check completed. No API key is required for this step.")


Python： 3.12.14
Working directory:  /workspace/scratch/37d8b59215b0/tmp/step1_notebook/execution/AI_Herding_Step1_Work
Environment check completed. No API key is required for this step.


## Code cell 2/7: Download the pinned project version

The download is automatic and does not require Git to be installed. The first run uses the network; a verified local copy is reused when available.


In [2]:
import io
import shutil
import urllib.request
import zipfile

REPO_COMMIT = "074faff3c8f6255c917d2a2593a5b476c947ca84"
ARCHIVE_URL = f"https://codeload.github.com/yuuhanjin/AI_herding_experiment/zip/{REPO_COMMIT}"
ARCHIVE_PATH = WORK_DIR / f"source_{REPO_COMMIT[:12]}.zip"
ORIGINAL_ROOT = WORK_DIR / "source_original"
EXPECTED_SOURCE_SHA256 = {'src/market/orders/order_state_manager.py': '705f379e7508695d6ac85edb1fdbd289a34483e8f41b51aa22e152f0895ac252', 'src/market/orders/order_service_factory.py': 'fb7ee88044be42ee56bde7c9efe6d06eebd8499d9f0878d2d116ef2d30532e0f', 'src/base_sim.py': '6404d996115ed0929460392abe6b152a5980df34cc91595028e7ee6f6a0754e5'}

if not ARCHIVE_PATH.exists() and not ORIGINAL_ROOT.exists():
    print("Downloading the team code...")
    with urllib.request.urlopen(ARCHIVE_URL, timeout=30) as response:
        archive_bytes = response.read()
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        if archive.testzip() is not None:
            raise RuntimeError("Downloaded ZIP checksum failed. Please rerun this cell.")
    ARCHIVE_PATH.write_bytes(archive_bytes)

if not ORIGINAL_ROOT.exists():
    extract_dir = WORK_DIR / "source_extract"
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir()
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        for entry in archive.infolist():
            target = (extract_dir / entry.filename).resolve()
            if not target.is_relative_to(extract_dir.resolve()):
                raise RuntimeError("Unsafe path detected in the ZIP; extraction stopped.")
        archive.extractall(extract_dir)
    candidates = [p for p in extract_dir.iterdir() if p.is_dir() and (p / "src").is_dir()]
    if len(candidates) != 1:
        raise RuntimeError("Cannot identify the project root. Retain the error details.")
    shutil.move(str(candidates[0]), str(ORIGINAL_ROOT))
    shutil.rmtree(extract_dir)

for relative_path, expected in EXPECTED_SOURCE_SHA256.items():
    path = ORIGINAL_ROOT / relative_path
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise RuntimeError(f"Original file differs from the pinned version: {relative_path}. Start again in a new working directory.")

parameters_path = ORIGINAL_ROOT / "data/runs/fig6_above_baseline/parameters.json"
parameters = json.loads(parameters_path.read_text(encoding="utf-8"))
CONFIGURED_LIMIT = int(parameters["AGENT_PARAMS"]["max_order_size"])
assert CONFIGURED_LIMIT == 1000

print("Code is ready: ", ORIGINAL_ROOT.resolve())
print("Key source hash checks: PASS")
print("Configured per-order limit: ", CONFIGURED_LIMIT, "shares")


Code is ready:  /workspace/scratch/37d8b59215b0/tmp/step1_notebook/execution/AI_Herding_Step1_Work/source_original
Key source hash checks: PASS
Configured per-order limit:  1000 shares


## Code cell 3/7: Prepare the offline test environment

Create a separate Python test environment automatically and install `numpy` and `pydantic` only if missing. This is not a model API call and does not require restarting the notebook.


In [3]:
import venv

TEST_ENV = WORK_DIR / "test_environment"
TEST_PYTHON = TEST_ENV / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not TEST_PYTHON.exists():
    print("Creating the test environment...")
    venv.EnvBuilder(with_pip=True, system_site_packages=True).create(TEST_ENV)

probe_code = r"""
import importlib.metadata, json
needed = []
try:
    import numpy
except (ImportError, OSError):
    needed.append('numpy>=1.26')
try:
    import pydantic
    version = tuple(int(x) for x in pydantic.__version__.split('.')[:2])
    if version < (2, 11) or version[0] != 2:
        needed.append('pydantic>=2.11,<3')
except (ImportError, OSError):
    needed.append('pydantic>=2.11,<3')
print(json.dumps(needed))
"""
probe = subprocess.run([str(TEST_PYTHON), "-c", probe_code], text=True, capture_output=True, check=True)
needed_packages = json.loads(probe.stdout)
if needed_packages:
    print("Installing test dependencies: ", ", ".join(needed_packages))
    install = subprocess.run(
        [str(TEST_PYTHON), "-m", "pip", "install", "--disable-pip-version-check", *needed_packages],
        text=True, capture_output=True,
    )
    if install.returncode:
        raise RuntimeError("Dependency installation failed:\n" + install.stdout[-2000:] + install.stderr[-2000:])
print("Test environment ready. No model API installation or configuration is required.")


Creating the test environment...
Test environment ready. No model API installation or configuration is required.


## Code cell 4/7: Inspect the original implementation

Run 32 cases: 16 quantity-boundary cases, 4 insufficient-resource cases, 8 non-positive-quantity cases, and 4 cases with alternative configured limits. Then run the repository's 3 existing matching tests.

The agents are manually configured test objects, and calling `make_decision()` is prohibited. Logging uses the repository's existing test doubles; order creation, resource handling, state management, and service factories use the project implementation.


In [4]:
AUDIT_PROGRAM = r"""import ast
import csv
import inspect
import json
import logging
import socket
import sys
from pathlib import Path
from types import SimpleNamespace

request = json.loads(sys.stdin.read())
root = Path(request['source']).resolve()
mode = request['mode']
output = Path(request['output']).resolve()
output.mkdir(parents=True, exist_ok=True)

sys.dont_write_bytecode = True
sys.path.insert(0, str(root / 'tests'))
import _logging_stub
_logging_stub.install()
logging.disable(logging.CRITICAL)

blocked_network_attempts = []
def deny_network(*args, **kwargs):
    blocked_network_attempts.append('blocked')
    raise RuntimeError('Network connections are disabled in this audit')
socket.socket.connect = deny_network
socket.socket.connect_ex = deny_network
socket.create_connection = deny_network

from agents.base_agent import BaseAgent
from agents.agent_manager.agent_repository import AgentRepository
from agents.agent_manager.services.agent_decision_service import AgentDecisionService
from market.orders.order_repository import OrderRepository
from market.orders.order_service_factory import OrderServiceFactory
from services.shared_service_factory import SharedServiceFactory

class OfflineAgent(BaseAgent):
    def make_decision(self, *args, **kwargs):
        raise AssertionError('Agent/model decision generation is not allowed in this audit')

has_limit_parameter = 'max_order_size' in inspect.signature(OrderServiceFactory.create_services).parameters
if mode == 'original':
    assert not has_limit_parameter, 'Original source unexpectedly already has a limit parameter'
else:
    assert has_limit_parameter, 'Patched factory is missing the limit parameter'

configuration_wired = any(
    isinstance(node, ast.Call)
    and isinstance(node.func, ast.Attribute)
    and node.func.attr == 'create_services'
    and any(keyword.arg == 'max_order_size' and 'agent_params' in ast.unparse(keyword.value)
            for keyword in node.keywords)
    for node in ast.walk(ast.parse((root / 'src/base_sim.py').read_text(encoding='utf-8')))
)
if mode == 'capped':
    assert configuration_wired, 'The scenario limit is not wired to the service factory'

def snapshot(agent):
    return {
        'available_cash': float(agent.available_cash),
        'committed_cash': float(agent.committed_cash),
        'available_shares': int(agent.positions.get('DEFAULT_STOCK', 0)),
        'committed_shares': int(agent.committed_positions.get('DEFAULT_STOCK', 0)),
    }

def audit_one(category, order_type, side, quantity, limit=1000,
              initial_cash=20000000, initial_shares=200000):
    agent = OfflineAgent('maker', initial_cash=initial_cash,
                         initial_shares=initial_shares, initial_price=56.0)
    context = SimpleNamespace(round_number=0, current_price=56.0)
    agent_repo = AgentRepository([agent], logging.getLogger('audit'), context)
    order_repo = OrderRepository()
    SharedServiceFactory.reset()
    SharedServiceFactory.initialize(order_book=object(), transaction_cost=0.0)
    kwargs = {'max_order_size': limit} if has_limit_parameter else {}
    state, execution_service = OrderServiceFactory.create_services(
        order_repository=order_repo, agent_repository=agent_repo,
        order_book=None, logger=logging.getLogger('audit'), **kwargs)
    if mode == 'capped':
        assert state.max_order_size == limit
    creator = AgentDecisionService(agent_repo, order_repo, state,
                                   logging.getLogger('audit'), logging.getLogger('audit'), context)
    before = snapshot(agent)
    order = None
    try:
        order = creator.create_order({
            'decision': side.title(), 'quantity': quantity,
            'order_type': order_type,
            'price_limit': 56.0 if order_type == 'limit' else None,
        }, 'maker')
        accepted, reason = state.handle_new_order(order, 56.0)
        final_state = order.state.value
    except ValueError as exc:
        accepted, reason, final_state = False, str(exc), 'creation_rejected'
    after = snapshot(agent)
    expected = quantity > 0 and category != 'resource_shortage'
    if mode == 'capped':
        expected = expected and quantity <= limit
    assert accepted == expected, (category, order_type, side, quantity, limit, accepted, reason)
    if not accepted:
        assert before == after, 'Rejected order changed resource commitments'
    if order is not None:
        assert order.filled_quantity == 0, 'This audit must not execute trades'
    if mode == 'capped' and quantity > limit:
        assert str(reason).startswith('MAX_ORDER_SIZE_EXCEEDED')
    row = {
        'mode': mode, 'category': category, 'order_type': order_type, 'side': side,
        'configured_limit': limit, 'requested_quantity': quantity,
        'accepted_quantity': order.quantity if accepted else 0,
        'accepted': bool(accepted), 'state': final_state,
        'order_id': order.order_id if order else '', 'reason': reason,
        'filled_quantity': order.filled_quantity if order else 0,
        'rejected_resources_unchanged': before == after if not accepted else '',
    }
    for key, value in before.items():
        row[key + '_before'] = value
    for key, value in after.items():
        row[key + '_after'] = value
    return row

rows = []
for kind in ('limit', 'market'):
    for side in ('buy', 'sell'):
        for quantity in (999, 1000, 1001, 100000):
            rows.append(audit_one('size_boundary', kind, side, quantity))
        rows.append(audit_one('resource_shortage', kind, side, 1000,
                              initial_cash=1 if side == 'buy' else 20000000,
                              initial_shares=0 if side == 'sell' else 200000))
        for quantity in (0, -1):
            rows.append(audit_one('nonpositive_quantity', kind, side, quantity))
for limit in (500, 2000):
    for quantity in (limit, limit + 1):
        rows.append(audit_one('config_override', 'limit', 'buy', quantity, limit=limit))
assert len(rows) == 32

# Existing project tests, executed against whichever source copy was supplied.
import test_crossed_market
regressions = []
for name in ('test_aggressive_limit_crossing', 'test_non_crossing_aggressive_limit',
             'test_partial_fill_aggressive_limit'):
    getattr(test_crossed_market, name)()
    regressions.append(name)
assert not blocked_network_attempts, 'An unexpected network request was attempted'

csv_path = output / f'order_size_{mode}.csv'
with csv_path.open('w', encoding='utf-8-sig', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
result = {
    'mode': mode, 'source': str(root), 'cases_completed': len(rows),
    'matching_regressions_passed': regressions,
    'scenario_configuration_wired': configuration_wired,
    'factory_supports_max_order_size': has_limit_parameter,
    'model_decision_calls': 0, 'network_attempts': len(blocked_network_attempts),
    'csv_path': str(csv_path), 'rows': rows,
}
(output / f'audit_{mode}.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(result, ensure_ascii=False))
"""


def run_audit(source_root, mode):
    payload = {"source": str(source_root.resolve()), "mode": mode, "output": str(RESULT_DIR.resolve())}
    process = subprocess.run(
        [str(TEST_PYTHON), "-c", AUDIT_PROGRAM],
        input=json.dumps(payload), text=True, capture_output=True,
        cwd=source_root, timeout=90,
    )
    if process.returncode:
        raise RuntimeError("Checks failed. Retain the following error:\n" + process.stdout[-3000:] + process.stderr[-5000:])
    return json.loads(process.stdout)


def display_table(rows, columns):
    try:
        from IPython.display import display, HTML
        from html import escape
        head = "".join(f"<th>{escape(str(label))}</th>" for key, label in columns)
        body = "".join(
            "<tr>" + "".join(f"<td>{escape(str(row.get(key, '')))}</td>" for key, label in columns) + "</tr>"
            for row in rows
        )
        display(HTML("<table style='border-collapse:collapse' border='1' cellpadding='7'><thead><tr>" + head + "</tr></thead><tbody>" + body + "</tbody></table>"))
    except ImportError:
        for row in rows:
            print({label: row.get(key, "") for key, label in columns})


BASELINE_RESULT = run_audit(ORIGINAL_ROOT, "original")
baseline_summary = []
for quantity in (999, 1000, 1001, 100000):
    subset = [r for r in BASELINE_RESULT["rows"] if r["category"] == "size_boundary" and r["requested_quantity"] == quantity]
    baseline_summary.append({"quantity": quantity, "accepted": f"{sum(r['accepted'] for r in subset)}/4", "meaning": "Within the configured size limit" if quantity <= CONFIGURED_LIMIT else "Oversized orders still pass initial validation"})
display_table(baseline_summary, [("quantity", "Requested shares"), ("accepted", "Accepted side/type combinations"), ("meaning", "Interpretation")])
print("Original implementation audit completed: 32 cases and 3 existing matching tests passed.")
print("Model decision calls: ", BASELINE_RESULT["model_decision_calls"])
print("Detailed results: ", BASELINE_RESULT["csv_path"])


Original implementation audit completed: 32 cases and 3 existing matching tests passed.
Model decision calls:  0
Detailed results:  /workspace/scratch/37d8b59215b0/tmp/step1_notebook/execution/AI_Herding_Step1_Work/results/order_size_original.csv


Requested shares,Accepted side/type combinations,Interpretation
999,4/4,Within the configured size limit
1000,4/4,Within the configured size limit
1001,4/4,Oversized orders still pass initial validation
100000,4/4,Oversized orders still pass initial validation


**How to read the table:** each quantity is checked for four combinations: limit buy, limit sell, market buy, and market sell.

If both 1001 and 100,000 shares show `4/4`, the size limit is not enforced on this initial-validation path. These checks establish acceptance and resource commitments; the tested orders are not matched.

## Code cell 5/7: Enforce the size limit in a separate copy

The policy is to reject oversized orders. Check the limit before committing resources and record the reason as `MAX_ORDER_SIZE_EXCEEDED`.

Modify three files in the test copy:

- `src/market/orders/order_state_manager.py`: add a configurable cap and the rejection logic.
- `src/market/orders/order_service_factory.py`: pass the cap to the state manager.
- `src/base_sim.py`: pass the scenario's `AGENT_PARAMS['max_order_size']` to the service factory.

Keep the original copy for comparison. The patch is not pushed to GitHub automatically. If the existing test copy contains other manual changes, stop rather than overwrite them.


In [5]:
import ast
import difflib

CAPPED_ROOT = WORK_DIR / "source_capped"
if not CAPPED_ROOT.exists():
    shutil.copytree(ORIGINAL_ROOT, CAPPED_ROOT)


def replace_once(text, old, new, filename):
    if text.count(old) != 1:
        raise RuntimeError(f"Source structure differs from the expected version: {filename}; automatic modification stopped.")
    return text.replace(old, new, 1)


updates = {}
state_file = "src/market/orders/order_state_manager.py"
original = (ORIGINAL_ROOT / state_file).read_text(encoding="utf-8")
updated = replace_once(
    original,
    "                 commitment_calculator: CommitmentCalculator):",
    "                 commitment_calculator: CommitmentCalculator,\n                 max_order_size: int | None = None):",
    state_file,
)
updated = replace_once(
    updated,
    "        self.commitment_calculator = commitment_calculator\n",
    "        self.commitment_calculator = commitment_calculator\n"
    "        if max_order_size is not None and (isinstance(max_order_size, bool) or not isinstance(max_order_size, int) or max_order_size <= 0):\n"
    "            raise ValueError('max_order_size must be a positive integer or None')\n"
    "        self.max_order_size = max_order_size\n",
    state_file,
)
updated = replace_once(
    updated,
    "        try:\n            # Calculate required commitment\n",
    "        try:\n"
    "            if self.max_order_size is not None and order.quantity > self.max_order_size:\n"
    "                message = (f'MAX_ORDER_SIZE_EXCEEDED: requested {order.quantity}, '\n"
    "                           f'limit {self.max_order_size}')\n"
    "                self.order_repository.transition_state(\n"
    "                    order.order_id, OrderState.CANCELLED, notes=message)\n"
    "                self.logger.warning(f'Order {order.order_id} rejected: {message}')\n"
    "                return False, message\n"
    "\n"
    "            # Calculate required commitment\n",
    state_file,
)
updates[state_file] = updated

factory_file = "src/market/orders/order_service_factory.py"
updated = (ORIGINAL_ROOT / factory_file).read_text(encoding="utf-8")
updated = replace_once(updated, "        logger\n    )", "        logger,\n        max_order_size: int | None = None\n    )", factory_file)
updated = replace_once(updated, "            commitment_calculator=commitment_calculator\n        )", "            commitment_calculator=commitment_calculator,\n            max_order_size=max_order_size\n        )", factory_file)
updates[factory_file] = updated

sim_file = "src/base_sim.py"
updated = (ORIGINAL_ROOT / sim_file).read_text(encoding="utf-8")
updated = replace_once(updated, "            logger=LoggingService.get_logger('order_state')\n        )", "            logger=LoggingService.get_logger('order_state'),\n            max_order_size=(agent_params or {}).get('max_order_size')\n        )", sim_file)
updates[sim_file] = updated

# Validate every intended change before writing any of the three files.
patch_parts = []
for relative_path, updated in updates.items():
    original = (ORIGINAL_ROOT / relative_path).read_text(encoding="utf-8")
    target = CAPPED_ROOT / relative_path
    if target.read_text(encoding="utf-8") not in (original, updated):
        raise RuntimeError(f"Test copy contains other changes: {relative_path}. Use another working directory to preserve both versions.")
    ast.parse(updated, filename=relative_path)
    patch_parts.extend(difflib.unified_diff(
        original.splitlines(keepends=True), updated.splitlines(keepends=True),
        fromfile="a/" + relative_path, tofile="b/" + relative_path,
    ))
for relative_path, updated in updates.items():
    (CAPPED_ROOT / relative_path).write_text(updated, encoding="utf-8")
for relative_path, expected in EXPECTED_SOURCE_SHA256.items():
    assert hashlib.sha256((ORIGINAL_ROOT / relative_path).read_bytes()).hexdigest() == expected

PATCH_PATH = RESULT_DIR / "step1_order_size_limit.patch"
PATCH_PATH.write_text("".join(patch_parts), encoding="utf-8")
print("Configurable cap connected in the test copy; all three source files passed syntax checks.")
print("Original-copy hash checks: PASS.")
print("Modified locations: ", CAPPED_ROOT.resolve())
print("Code patch: ", PATCH_PATH.resolve())


Configurable cap connected in the test copy; all three source files passed syntax checks.
Original-copy hash checks: PASS.
Modified locations:  /workspace/scratch/37d8b59215b0/tmp/step1_notebook/execution/AI_Herding_Step1_Work/source_capped
Code patch:  /workspace/scratch/37d8b59215b0/tmp/step1_notebook/execution/AI_Herding_Step1_Work/results/step1_order_size_limit.patch


## Code cell 6/7: Validate the patch and compare implementations

Rerun the same 32 cases and 3 existing matching tests. Oversized orders must be rejected without changing resource commitments. Alternative limits of 500 or 2000 must also be enforced according to their configuration.


In [6]:
CAPPED_RESULT = run_audit(CAPPED_ROOT, "capped")
COMPARISON = []
for quantity in (999, 1000, 1001, 100000):
    before = [r for r in BASELINE_RESULT["rows"] if r["category"] == "size_boundary" and r["requested_quantity"] == quantity]
    after = [r for r in CAPPED_RESULT["rows"] if r["category"] == "size_boundary" and r["requested_quantity"] == quantity]
    accepted_before = sum(r["accepted"] for r in before)
    accepted_after = sum(r["accepted"] for r in after)
    assert accepted_after == (4 if quantity <= CONFIGURED_LIMIT else 0)
    COMPARISON.append({
        "requested_quantity": quantity,
        "original_accepted": f"{accepted_before}/4",
        "capped_accepted": f"{accepted_after}/4",
        "result": "PASS",
    })

display_table(COMPARISON, [
    ("requested_quantity", "Requested shares"), ("original_accepted", "Accepted in original implementation"),
    ("capped_accepted", "Accepted with cap enforced"), ("result", "Patch check"),
])

comparison_path = RESULT_DIR / "order_size_comparison.csv"
with comparison_path.open("w", encoding="utf-8-sig", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(COMPARISON[0]))
    writer.writeheader()
    writer.writerows(COMPARISON)

assert CAPPED_RESULT["cases_completed"] == 32
assert len(CAPPED_RESULT["matching_regressions_passed"]) == 3
assert CAPPED_RESULT["scenario_configuration_wired"]
assert CAPPED_RESULT["model_decision_calls"] == 0
assert CAPPED_RESULT["network_attempts"] == 0
print("Patch validation completed: 32 cases behaved as expected and 3 existing matching tests passed.")
print("Insufficient resources and non-positive quantities are still rejected; the cap follows configuration changes.")
print("Available and committed cash and shares remain unchanged for every rejected order.")
print("Model calls: 0; order-test network requests: 0.")


Patch validation completed: 32 cases behaved as expected and 3 existing matching tests passed.
Insufficient resources and non-positive quantities are still rejected; the cap follows configuration changes.
Available and committed cash and shares remain unchanged for every rejected order.
Model calls: 0; order-test network requests: 0.


Requested shares,Accepted in original implementation,Accepted with cap enforced,Patch check
999,4/4,4/4,PASS
1000,4/4,4/4,PASS
1001,4/4,0/4,PASS
100000,4/4,0/4,PASS


## Code cell 7/7: Save reportable results

After a successful run, use the printed description to report this progress. The checks do not identify how the patch changes market trading volume; that requires subsequent order replay or controlled experiments.


In [7]:
summary = {
    "repository_commit": REPO_COMMIT,
    "policy": "Reject oversized orders before committing resources",
    "configured_limit": CONFIGURED_LIMIT,
    "boundary_and_resource_cases_per_implementation": 32,
    "existing_matching_regressions_per_implementation": 3,
    "model_api_calls": 0,
    "original_csv": BASELINE_RESULT["csv_path"],
    "capped_csv": CAPPED_RESULT["csv_path"],
    "comparison_csv": str(comparison_path.resolve()),
    "patch": str(PATCH_PATH.resolve()),
    "scope": "Order creation and initial resource validation; no full LLM market run",
}
(RESULT_DIR / "step1_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("Reportable progress: ")
print("This stage audited quantity constraints with manually constructed orders. With sufficient resources, the original implementation allowed 1001-share and 100,000-share orders through initial validation.")
print("In a separate test copy, the scenario cap was connected to order validation using an oversized-order rejection policy. After the patch, 999/1000-share orders passed, 1001/100,000-share orders were rejected, and rejection committed no resources.")
print("Each implementation completed 32 cases and 3 existing matching regression tests. This validates initial order processing; it does not constitute a new 15-round experiment or identify trading-volume effects.")
print("\nOutput directory: ", RESULT_DIR.resolve())
for name in ["order_size_original.csv", "order_size_capped.csv", "order_size_comparison.csv", "step1_order_size_limit.patch", "step1_summary.json"]:
    print(" -", name)
print("\nInspect the comparison table in cell 6 first, then continue to Next Step 2: order lifecycle and resource logs.")


Reportable progress: 
This stage audited quantity constraints with manually constructed orders. With sufficient resources, the original implementation allowed 1001-share and 100,000-share orders through initial validation.
In a separate test copy, the scenario cap was connected to order validation using an oversized-order rejection policy. After the patch, 999/1000-share orders passed, 1001/100,000-share orders were rejected, and rejection committed no resources.
Each implementation completed 32 cases and 3 existing matching regression tests. This validates initial order processing; it does not constitute a new 15-round experiment or identify trading-volume effects.

Output directory:  /workspace/scratch/37d8b59215b0/tmp/step1_notebook/execution/AI_Herding_Step1_Work/results
 - order_size_original.csv
 - order_size_capped.csv
 - order_size_comparison.csv
 - step1_order_size_limit.patch
 - step1_summary.json

Inspect the comparison table in cell 6 first, then continue to Next Step 2: or

### Troubleshooting

- **Unsupported Python version:** switch to a Python 3.11 or later kernel.
- **Download failure:** check GitHub access and rerun code cell 2.
- **Dependency installation failure:** retain the error from cell 3 and resolve the environment issue first.
- **Failed checks:** retain the complete error and do not report a research conclusion from the failed run.
- **Repeated execution:** rerun in order. The original source hashes are checked again, and an already-correct patched copy can be reused.

There is no need to run `run_prompt_hierarchy_probe.py`, `run_prompt_contrast.py`, or an LLM scenario for this step. Only manually constructed orders are used to inspect the quantity constraint.
